# Preparação para Modelagem — Base Municipal de Alfabetização

Este notebook inicia a etapa posterior à EDA e **termina antes do treinamento de qualquer modelo**.

As decisões de qualidade, granularidade, targets e prevenção de leakage já foram estabelecidas no notebook `1_0_eda_base_analitica_DMN_EDA_final_auditavel_corrigido.ipynb`. Aqui elas são apenas aplicadas para produzir bases prontas para a modelagem.

**Saídas esperadas:** base `município-ano`, recortes temporais 2024/2025, matrizes `X/y` por problema supervisionado **ainda não imputadas** e regras de preparação registradas para uso no próximo notebook. A imputação será ajustada dentro de cada fold de validação de 2024 e, após a escolha do modelo, novamente sobre 100% de 2024 antes do teste temporal final em 2025.


## 1. Ambiente, dependências e leitura

A modelagem utiliza **2024 como desenvolvimento/validação** e **2025 como teste temporal final**. A partição de 2023 não é carregada porque não possui o histórico `_lag1` necessário para o mesmo conjunto de features.

Para reduzir o uso de memória, cada partição aluno-ano é lida, filtrada, agregada para município-ano e descartada antes da leitura da próxima.

In [20]:
AMBIENTE_COLAB = False

In [21]:
if AMBIENTE_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')


In [22]:
# Dependências necessárias no ambiente Colab.
if AMBIENTE_COLAB:
    !pip install -q loguru python-dotenv pyarrow scikit-learn joblib

In [23]:
import gc
import sys
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

if AMBIENTE_COLAB:
    PROJECT_ROOT = Path('/content/drive/Othercomputers/@lua2026/01_projects/2026_FIAP/desafios/postech-challenge-3')
else:
    PROJECT_ROOT = Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import PROCESSED_DATA_DIR

CAMINHO_BASE = PROCESSED_DATA_DIR / 'base_analitica'
ANOS_MODELAGEM = [2024, 2025]
ARQUIVOS_ANO = {
    ano: CAMINHO_BASE / f'ano={ano}' / 'part.parquet'
    for ano in ANOS_MODELAGEM
}

for ano, arquivo in ARQUIVOS_ANO.items():
    if not arquivo.exists():
        raise FileNotFoundError(f'Arquivo não encontrado para {ano}: {arquivo}')

print('Partições localizadas:')
for ano, arquivo in ARQUIVOS_ANO.items():
    print(f'  {ano}: {arquivo}')

gc.collect()


Partições localizadas:
  2024: /home/mburger/study/postech/techchallenge3/postech-challenge-3/data/processed/base_analitica/ano=2024/part.parquet
  2025: /home/mburger/study/postech/techchallenge3/postech-challenge-3/data/processed/base_analitica/ano=2025/part.parquet


0

## 2. Decisões da EDA aplicadas à base

A lista abaixo congela a seleção já auditada na EDA: 12 features removidas por missingness superior a 10% em 2024 e 27 por redundância municipal (`|Spearman| ≥ 0,90`). Não há nova seleção de features neste notebook.

In [24]:
# ------------------------------------------------------------
# Decisões de qualidade já aprovadas na EDA
# ------------------------------------------------------------

REMOVER_MISSING_10 = [
    'ctx_atu_fundamental_6_ano',
    'ctx_atu_fundamental_7_ano',
    'ctx_atu_fundamental_8_ano',
    'ctx_atu_fundamental_9_ano',
    'ctx_atu_fundamental_anos_finais',
    'ctx_atu_medio',
    'ctx_atu_medio_1_serie',
    'ctx_atu_medio_2_serie',
    'ctx_atu_medio_3_serie',
    'ctx_atu_medio_4_serie',
    'ctx_atu_medio_nao_seriado',
    'ctx_atu_multietapa',
]

REMOVER_REDUNDANCIA = [
    'ctx_atlas_idhm',
    'ctx_atlas_idhm_educacao',
    'ctx_atlas_idhm_renda',
    'ctx_atlas_indice_theil',
    'ctx_atlas_mortalidade_ate_1_ano',
    'ctx_atlas_mortalidade_ate_5_anos',
    'ctx_atlas_percentual_extremamente_pobres',
    'ctx_atlas_percentual_extremamente_pobres_criancas',
    'ctx_atlas_percentual_pobres',
    'ctx_atlas_percentual_pobres_criancas',
    'ctx_atlas_percentual_vulneraveis_pobreza',
    'ctx_atlas_percentual_vulneraveis_pobreza_criancas',
    'ctx_atlas_razao_10_ricos_40_pobres',
    'ctx_atlas_renda_per_capita',
    'ctx_atlas_taxa_criancas_em_domicilios_sem_fundamental',
    'ctx_atlas_taxa_domicilios_vulneraveis_sem_fundamental',
    'ctx_atlas_taxa_frequencia_4a5',
    'ctx_atlas_taxa_frequencia_6a14',
    'ctx_atlas_taxa_fundamental_incompleto',
    'ctx_atu_fundamental',
    'ctx_fundeb_nse_municipio',
    'ctx_fundeb_ponderador_nse_municipio',
    'ctx_fundeb_ponderador_nse_uf',
    'ctx_ibge_renda_domiciliar_per_capita_mediana',
    'ctx_inep_mun_nivel_alfabetizacao_lag1',
    'ctx_inep_uf_meta',
    'ctx_inep_uf_taxa_alfabetizacao_lag1',
]

COLUNAS_REMOVER = sorted(set(REMOVER_MISSING_10) | set(REMOVER_REDUNDANCIA))

COLUNAS_OBRIGATORIAS = {
    'ano', 'id_uf', 'id_municipio', 'id_escola',
    'dependencia_administrativa', 'label_alfabetizado', 'label_proficiencia',
    'ctx_inep_mun_meta',
    'ctx_inep_mun_taxa_alfabetizacao_lag1',
    'ctx_inep_mun_percentual_participacao_lag1',
    'ctx_inep_mun_media_portugues_lag1',
    'ctx_ibge_populacao_residente',
}

assert len(REMOVER_MISSING_10) == 12
assert len(REMOVER_REDUNDANCIA) == 27
assert len(COLUNAS_REMOVER) == 39
assert not (COLUNAS_OBRIGATORIAS & set(COLUNAS_REMOVER))

print(f'Colunas removidas conforme EDA: {len(COLUNAS_REMOVER)}')
print(f'  missingness >10% em 2024: {len(REMOVER_MISSING_10)}')
print(f'  redundância: {len(REMOVER_REDUNDANCIA)}')

gc.collect()

Colunas removidas conforme EDA: 39
  missingness >10% em 2024: 12
  redundância: 27


0

## 3. Reconstrução da unidade município-ano

Cada arquivo ainda está na granularidade aluno-ano. A função abaixo aplica a seleção da EDA e colapsa cada partição para a chave **`ano + id_uf + id_municipio`**.

As variáveis `alunos`, `escolas` e `pct_rede_municipal` são mantidas como metadados/auditoria da observação, mas não entram automaticamente em `X`, pois são derivadas da composição da avaliação corrente.

In [25]:
ALVO_ALUNO_CLASSIFICACAO = 'label_alfabetizado'
ALVO_ALUNO_REGRESSAO = 'label_proficiencia'
CHAVES_MUNICIPIO_ANO = ['ano', 'id_uf', 'id_municipio']

ALIASES_MUNICIPIO = {
    'ctx_inep_mun_meta': 'meta',
    'ctx_inep_mun_taxa_alfabetizacao_lag1': 'taxa_lag1',
    'ctx_inep_mun_percentual_participacao_lag1': 'participacao_lag1',
    'ctx_inep_mun_media_portugues_lag1': 'proficiencia_lag1',
    'ctx_ibge_populacao_residente': 'populacao',
}


def agregar_municipio_ano(arquivo):
    """Lê uma partição aluno-ano, aplica decisões da EDA e retorna município-ano."""
    parte = pd.read_parquet(arquivo)
    parte = parte.drop(columns=['_gold_processed_at'], errors='ignore')

    faltantes = sorted(COLUNAS_OBRIGATORIAS - set(parte.columns))
    if faltantes:
        raise KeyError(f'Colunas obrigatórias ausentes em {arquivo}: {faltantes}')

    parte = parte.drop(columns=COLUNAS_REMOVER, errors='ignore')
    colunas_ctx = [c for c in parte.columns if c.startswith('ctx_')]

    resumo = parte.groupby(CHAVES_MUNICIPIO_ANO, observed=True).agg(
        alunos=(ALVO_ALUNO_CLASSIFICACAO, 'size'),
        taxa_realizada=(ALVO_ALUNO_CLASSIFICACAO, 'mean'),
        proficiencia_media=(ALVO_ALUNO_REGRESSAO, 'mean'),
        escolas=('id_escola', 'nunique'),
        pct_rede_municipal=(
            'dependencia_administrativa',
            lambda s: (s == 3).mean()
        ),
    )

    resumo['taxa_realizada'] = resumo['taxa_realizada'].astype(float) * 100
    resumo['pct_rede_municipal'] = resumo['pct_rede_municipal'].astype(float) * 100

    contexto = (
        parte.groupby(CHAVES_MUNICIPIO_ANO, observed=True)[colunas_ctx]
        .mean()
    )

    resultado = resumo.join(contexto).reset_index()

    del parte, resumo, contexto, colunas_ctx
    gc.collect()

    return resultado


partes_municipais = []
for ano in ANOS_MODELAGEM:
    print(f'Agregando {ano}...')
    parte_municipal = agregar_municipio_ano(ARQUIVOS_ANO[ano])
    partes_municipais.append(parte_municipal)
    print(f'  {len(parte_municipal):,} municípios')

municipios_ano = pd.concat(partes_municipais, ignore_index=True)
del partes_municipais, parte_municipal
gc.collect()

faltantes_alias = [c for c in ALIASES_MUNICIPIO if c not in municipios_ano.columns]
if faltantes_alias:
    raise KeyError(f'Colunas necessárias aos aliases não preservadas: {faltantes_alias}')

municipios_ano = municipios_ano.rename(columns=ALIASES_MUNICIPIO)

assert not municipios_ano.duplicated(CHAVES_MUNICIPIO_ANO).any()

print(f'Base município-ano: {municipios_ano.shape[0]:,} linhas x {municipios_ano.shape[1]} colunas')
display(municipios_ano.groupby('ano', observed=True).size().rename('municipios').to_frame())

gc.collect()

Agregando 2024...


  5,517 municípios
Agregando 2025...
  5,556 municípios
Base município-ano: 11,073 linhas x 45 colunas


,municipios
ano,
2024,5517
2025,5556


0

## 4. Targets municipais e variáveis derivadas permitidas

Os targets são construídos somente depois da mudança de granularidade. `taxa_realizada`, `proficiencia_media`, `nivel_realizado`, os targets binários e `distancia_ate_a_meta` descrevem o resultado corrente e são proibidos em `X`.

`esforco_pactuado = meta - taxa_lag1` é permitido para Q4 porque usa apenas informação disponível antes do desfecho corrente.

In [26]:
ALVO_REGRESSAO_MUNICIPIO = 'taxa_realizada'
ALVO_CLASSIFICACAO_RISCO = 'risco_educacional'
ALVO_CLASSIFICACAO_META = 'risco_nao_atingir_meta'
LIMIAR_RISCO_EDUCACIONAL = 50.0

municipios_ano[ALVO_CLASSIFICACAO_RISCO] = np.where(
    municipios_ano[ALVO_REGRESSAO_MUNICIPIO].isna(),
    np.nan,
    (municipios_ano[ALVO_REGRESSAO_MUNICIPIO] < LIMIAR_RISCO_EDUCACIONAL).astype(float)
)

municipios_ano[ALVO_CLASSIFICACAO_META] = np.where(
    municipios_ano['meta'].isna() | municipios_ano[ALVO_REGRESSAO_MUNICIPIO].isna(),
    np.nan,
    (municipios_ano[ALVO_REGRESSAO_MUNICIPIO] < municipios_ano['meta']).astype(float)
)

municipios_ano['distancia_ate_a_meta'] = (
    municipios_ano['taxa_realizada'] - municipios_ano['meta']
)

municipios_ano['esforco_pactuado'] = (
    municipios_ano['meta'] - municipios_ano['taxa_lag1']
)

COLUNAS_PROIBIDAS_X = {
    'taxa_realizada',
    'proficiencia_media',
    'risco_educacional',
    'risco_nao_atingir_meta',
    'distancia_ate_a_meta',
    # derivados/composição da avaliação corrente
    'alunos',
    'escolas',
    'pct_rede_municipal',
}

print('Targets municipais definidos.')
print('Q1:', ALVO_REGRESSAO_MUNICIPIO)
print('Q2:', ALVO_CLASSIFICACAO_RISCO)
print('Q4:', ALVO_CLASSIFICACAO_META)

gc.collect()

Targets municipais definidos.
Q1: taxa_realizada
Q2: risco_educacional
Q4: risco_nao_atingir_meta


0

## 5. Conjunto de features disponível antes do desfecho

A preparação separa o conjunto estrutural/histórico das variáveis de meta. Isso permite que Q1 e Q2 sejam avaliados sem transformar a meta pactuada em proxy do resultado, enquanto Q4 inclui explicitamente `meta` e `esforco_pactuado`.

`id_uf` e `id_municipio` permanecem apenas como chaves de rastreabilidade; não entram como números contínuos no modelo.

In [27]:
COLUNAS_CHAVE = ['ano', 'id_uf', 'id_municipio']
COLUNAS_META = ['meta', 'esforco_pactuado']

# Depois dos aliases, todo contexto retido continua com prefixo ctx_,
# exceto as cinco colunas renomeadas abaixo.
ALIASES_FEATURES = [
    'taxa_lag1',
    'participacao_lag1',
    'proficiencia_lag1',
    'populacao',
]

features_contexto = [
    c for c in municipios_ano.columns
    if c.startswith('ctx_')
]

FEATURES_ESTRUTURAIS_HISTORICAS = sorted(set(
    features_contexto + ALIASES_FEATURES
) - COLUNAS_PROIBIDAS_X)

FEATURES_Q1 = FEATURES_ESTRUTURAIS_HISTORICAS.copy()
FEATURES_Q2 = FEATURES_ESTRUTURAIS_HISTORICAS.copy()
FEATURES_Q2_COM_META = FEATURES_Q2 + ['meta']
FEATURES_Q4 = FEATURES_ESTRUTURAIS_HISTORICAS + COLUNAS_META
FEATURES_Q3 = FEATURES_ESTRUTURAIS_HISTORICAS.copy()

for nome, features in {
    'Q1': FEATURES_Q1,
    'Q2': FEATURES_Q2,
    'Q2_com_meta': FEATURES_Q2_COM_META,
    'Q3': FEATURES_Q3,
    'Q4': FEATURES_Q4,
}.items():
    proibidas = set(features) & COLUNAS_PROIBIDAS_X
    assert not proibidas, f'{nome} contém leakage/pós-desfecho: {sorted(proibidas)}'
    faltantes = set(features) - set(municipios_ano.columns)
    assert not faltantes, f'{nome} contém features ausentes: {sorted(faltantes)}'
    print(f'{nome}: {len(features)} features')

del features_contexto
gc.collect()

Q1: 36 features
Q2: 36 features
Q2_com_meta: 37 features
Q3: 36 features
Q4: 38 features


0

## 6. Separação temporal antes de qualquer imputação

A separação temporal é feita antes de qualquer estatística de preenchimento:

- **2024** é o conjunto de desenvolvimento. A validação de modelos será feita somente dentro de 2024, preferencialmente por cross-validation;
- **2025** é reservado como **teste temporal final** e não deve participar de seleção de modelo, hiperparâmetros, threshold ou imputação durante o desenvolvimento.

Os nulos das features são preservados neste notebook. A imputação será parte do `Pipeline` de modelagem, garantindo que a mediana seja aprendida apenas no subconjunto de treino de cada fold.


In [28]:
ANO_DESENVOLVIMENTO = 2024
ANO_TESTE_FINAL = 2025

base_dev = municipios_ano.loc[
    municipios_ano['ano'].eq(ANO_DESENVOLVIMENTO)
].copy()

base_teste_final = municipios_ano.loc[
    municipios_ano['ano'].eq(ANO_TESTE_FINAL)
].copy()

assert len(base_dev) > 0 and len(base_teste_final) > 0
assert base_dev['ano'].eq(ANO_DESENVOLVIMENTO).all()
assert base_teste_final['ano'].eq(ANO_TESTE_FINAL).all()

print(
    f'Desenvolvimento ({ANO_DESENVOLVIMENTO}): '
    f'{len(base_dev):,} municípios'
)
print(
    f'Teste temporal final ({ANO_TESTE_FINAL}): '
    f'{len(base_teste_final):,} municípios'
)

gc.collect()


Desenvolvimento (2024): 5,517 municípios
Teste temporal final (2025): 5,556 municípios


0

## 7. Nulos remanescentes — regra para o pipeline de modelagem

Os nulos remanescentes **não são preenchidos neste notebook**.

A regra definida para a próxima etapa é:

- features numéricas: `SimpleImputer(strategy='median', add_indicator=True)`;
- o imputador deve ficar **dentro do `Pipeline` do modelo**;
- em cada fold de validação de 2024, o imputador é ajustado somente no subconjunto de treino daquele fold;
- após escolher a configuração final, o pipeline é ajustado novamente em **100% de 2024**;
- somente então o pipeline é aplicado uma única vez em **2025**;
- targets ausentes nunca são imputados.

A célula abaixo apenas registra essa configuração e prepara as matrizes brutas, preservando os `NaN`.


In [29]:
# Configuração que será usada dentro dos pipelines de modelagem.
# Nenhum imputador é ajustado neste notebook.
CONFIG_IMPUTACAO_NUMERICA = {
    'strategy': 'median',
    'add_indicator': True,
    'keep_empty_features': True,
}


def preparar_X_bruto(base_dev, base_teste_final, features, nome):
    """Prepara X numérico sem imputar e audita os nulos por período."""
    X_dev = (
        base_dev[features]
        .apply(pd.to_numeric, errors='coerce')
        .copy()
    )

    X_teste_final = (
        base_teste_final[features]
        .apply(pd.to_numeric, errors='coerce')
        .copy()
    )

    nulos_dev = int(X_dev.isna().sum().sum())
    nulos_teste = int(X_teste_final.isna().sum().sum())

    print(
        f'{nome}: {len(features)} features | '
        f'nulos preservados: dev_2024={nulos_dev:,}, '
        f'teste_2025={nulos_teste:,}'
    )

    gc.collect()
    return X_dev, X_teste_final


X_dev_q1, X_teste_q1 = preparar_X_bruto(
    base_dev, base_teste_final, FEATURES_Q1, 'Q1'
)

X_dev_q2, X_teste_q2 = preparar_X_bruto(
    base_dev, base_teste_final, FEATURES_Q2, 'Q2'
)

X_dev_q2_meta, X_teste_q2_meta = preparar_X_bruto(
    base_dev, base_teste_final, FEATURES_Q2_COM_META, 'Q2 + meta'
)

X_dev_q4, X_teste_q4 = preparar_X_bruto(
    base_dev, base_teste_final, FEATURES_Q4, 'Q4'
)

X_dev_q3, X_teste_q3 = preparar_X_bruto(
    base_dev, base_teste_final, FEATURES_Q3, 'Q3'
)

gc.collect()


Q1: 36 features | nulos preservados: dev_2024=837, teste_2025=858
Q2: 36 features | nulos preservados: dev_2024=837, teste_2025=858
Q2 + meta: 37 features | nulos preservados: dev_2024=912, teste_2025=915
Q4: 38 features | nulos preservados: dev_2024=1,033, teste_2025=1,041
Q3: 36 features | nulos preservados: dev_2024=837, teste_2025=858


0

## 8. Matrizes finais por problema — ainda sem imputação

As matrizes abaixo preservam os índices da base municipal para manter `X`, `y` e chaves alinhados. Quando o target não existe, a linha é removida **apenas daquele problema**.

Os `NaN` em `X` permanecem intencionalmente. Eles serão tratados dentro do `Pipeline` durante a validação em 2024, evitando leakage entre folds.


In [30]:
def montar_dataset_supervisionado(
    base_dev,
    base_teste_final,
    X_dev,
    X_teste_final,
    alvo,
    nome,
):
    idx_dev = base_dev.index[base_dev[alvo].notna()]
    idx_teste = base_teste_final.index[base_teste_final[alvo].notna()]

    pacote = {
        'X_dev': X_dev.loc[idx_dev].copy(),
        'y_dev': base_dev.loc[idx_dev, alvo].copy(),
        'X_teste_final': X_teste_final.loc[idx_teste].copy(),
        'y_teste_final': base_teste_final.loc[idx_teste, alvo].copy(),
        'chaves_dev': base_dev.loc[idx_dev, COLUNAS_CHAVE].copy(),
        'chaves_teste_final': base_teste_final.loc[idx_teste, COLUNAS_CHAVE].copy(),
    }

    assert pacote['X_dev'].index.equals(pacote['y_dev'].index)
    assert pacote['X_teste_final'].index.equals(pacote['y_teste_final'].index)

    print(
        f'{nome}: desenvolvimento={len(idx_dev):,} | '
        f'teste_final={len(idx_teste):,} | '
        f'features={pacote["X_dev"].shape[1]}'
    )
    return pacote


dados_q1 = montar_dataset_supervisionado(
    base_dev, base_teste_final,
    X_dev_q1, X_teste_q1,
    ALVO_REGRESSAO_MUNICIPIO,
    'Q1 — regressão'
)

dados_q2 = montar_dataset_supervisionado(
    base_dev, base_teste_final,
    X_dev_q2, X_teste_q2,
    ALVO_CLASSIFICACAO_RISCO,
    'Q2 — risco educacional'
)

# Variante de sensibilidade já preparada, mas ainda não comparada/modelada.
dados_q2_com_meta = montar_dataset_supervisionado(
    base_dev, base_teste_final,
    X_dev_q2_meta, X_teste_q2_meta,
    ALVO_CLASSIFICACAO_RISCO,
    'Q2 — risco educacional + meta'
)

dados_q4 = montar_dataset_supervisionado(
    base_dev, base_teste_final,
    X_dev_q4, X_teste_q4,
    ALVO_CLASSIFICACAO_META,
    'Q4 — não atingir meta'
)

# Q3 não possui target.
dados_q3 = {
    'X_dev': X_dev_q3.copy(),
    'X_teste_final': X_teste_q3.copy(),
    'chaves_dev': base_dev[COLUNAS_CHAVE].copy(),
    'chaves_teste_final': base_teste_final[COLUNAS_CHAVE].copy(),
}

print(
    f'Q3 — clustering: desenvolvimento={len(dados_q3["X_dev"]):,} | '
    f'teste_final={len(dados_q3["X_teste_final"]):,} | '
    f'features={dados_q3["X_dev"].shape[1]}'
)

gc.collect()


Q1 — regressão: desenvolvimento=5,517 | teste_final=5,556 | features=36
Q2 — risco educacional: desenvolvimento=5,517 | teste_final=5,556 | features=36
Q2 — risco educacional + meta: desenvolvimento=5,517 | teste_final=5,556 | features=37
Q4 — não atingir meta: desenvolvimento=5,442 | teste_final=5,499 | features=38
Q3 — clustering: desenvolvimento=5,517 | teste_final=5,556 | features=36


0

## 9. Auditoria final da preparação

Esta célula verifica as condições mínimas antes de iniciar qualquer algoritmo:

- separação temporal íntegra;
- ausência de leakage explícito;
- targets válidos e não imputados;
- `X` e `y` alinhados;
- nulos em `X` **preservados intencionalmente** para tratamento dentro dos pipelines de modelagem.


In [31]:
pacotes_supervisionados = {
    'Q1': dados_q1,
    'Q2': dados_q2,
    'Q2_com_meta': dados_q2_com_meta,
    'Q4': dados_q4,
}

linhas_auditoria = []

for nome, pacote in pacotes_supervisionados.items():
    Xdev = pacote['X_dev']
    Xte = pacote['X_teste_final']
    ydev = pacote['y_dev']
    yte = pacote['y_teste_final']

    assert not ydev.isna().any()
    assert not yte.isna().any()
    assert pacote['chaves_dev']['ano'].eq(ANO_DESENVOLVIMENTO).all()
    assert pacote['chaves_teste_final']['ano'].eq(ANO_TESTE_FINAL).all()
    assert not (set(Xdev.columns) & COLUNAS_PROIBIDAS_X)
    assert Xdev.index.equals(ydev.index)
    assert Xte.index.equals(yte.index)

    linhas_auditoria.append({
        'problema': nome,
        'n_dev_2024': len(Xdev),
        'n_teste_final_2025': len(Xte),
        'features': Xdev.shape[1],
        'nulos_X_dev': int(Xdev.isna().sum().sum()),
        'nulos_X_teste_final': int(Xte.isna().sum().sum()),
        'nulos_y_dev': int(ydev.isna().sum()),
        'nulos_y_teste_final': int(yte.isna().sum()),
    })

# Q3
assert dados_q3['chaves_dev']['ano'].eq(ANO_DESENVOLVIMENTO).all()
assert dados_q3['chaves_teste_final']['ano'].eq(ANO_TESTE_FINAL).all()
assert not (set(dados_q3['X_dev'].columns) & COLUNAS_PROIBIDAS_X)

resumo_preparacao = pd.DataFrame(linhas_auditoria).set_index('problema')
display(resumo_preparacao)

print('Auditoria final: OK')
print('Nulos em X foram preservados para imputação dentro do pipeline.')
print('Nenhum modelo foi treinado neste notebook.')

del linhas_auditoria
gc.collect()


,n_dev_2024,n_teste_final_2025,features,nulos_X_dev,nulos_X_teste_final,nulos_y_dev,nulos_y_teste_final
problema,,,,,,,
Q1,5517,5556,36,837,858,0,0
Q2,5517,5556,36,837,858,0,0
Q2_com_meta,5517,5556,37,912,915,0,0
Q4,5442,5499,38,509,622,0,0


Auditoria final: OK
Nulos em X foram preservados para imputação dentro do pipeline.
Nenhum modelo foi treinado neste notebook.


0

## 10. Nome dos municípios — API de Localidades do IBGE

A base município-ano identifica cada município apenas pelo código IBGE (`id_municipio`). Para tornar os resultados legíveis, a lista oficial de municípios é consultada na **API de Localidades do IBGE** (`/api/v1/localidades/municipios`) e convertida na tabela `id_municipio → nome_municipio`, incorporada por `merge` à base que será gravada em `ARQUIVO_DADOS_MODELAGEM`.

- o `merge` usa o código IBGE de 7 dígitos (`how='left'`, `validate='many_to_one'`), nunca o nome, pois há municípios homônimos em UFs diferentes;
- `nome_municipio` é coluna de **identificação**, como `id_municipio`, e não entra em `X`: as listas de features são explícitas e as matrizes das seções 7 e 8 já foram montadas;
- `municipios_ano` só é substituída se o número de linhas não mudar e todos os municípios receberem nome; caso contrário, a célula falha.


In [32]:
# ------------------------------------------------------------
# Nome dos municípios via API de Localidades do IBGE
# ------------------------------------------------------------
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

URL_API_MUNICIPIOS_IBGE = (
    'https://servicodados.ibge.gov.br/api/v1/localidades/municipios'
)


def consultar_municipios_ibge(url=URL_API_MUNICIPIOS_IBGE, timeout=60):
    """Retorna a lista de municípios da API de Localidades do IBGE."""
    retentativas = Retry(
        total=3,
        backoff_factor=2,
        status_forcelist=(429, 500, 502, 503, 504),
    )
    with requests.Session() as sessao:
        sessao.mount('https://', HTTPAdapter(max_retries=retentativas))
        resposta = sessao.get(url, timeout=timeout)
        resposta.raise_for_status()
        municipios = resposta.json()

    if not isinstance(municipios, list) or not municipios:
        raise ValueError('Resposta inesperada da API de Localidades do IBGE.')
    return municipios


def normalizar_codigo_ibge(serie):
    """Converte o código IBGE (int, float, texto ou category) para Int64."""
    return (
        pd.to_numeric(serie.astype('string').str.strip(), errors='coerce')
        .astype('Int64')
    )


# Tabela id_municipio → nome_municipio.
# Só 'id' e 'nome' são lidos: os níveis aninhados (microrregião etc.)
# podem vir nulos para municípios instalados recentemente.
tabela_municipios_ibge = pd.DataFrame(
    [
        {'id_municipio': m['id'], 'nome_municipio': m['nome']}
        for m in consultar_municipios_ibge()
    ]
)
tabela_municipios_ibge['id_municipio'] = normalizar_codigo_ibge(
    tabela_municipios_ibge['id_municipio']
)
tabela_municipios_ibge['nome_municipio'] = (
    tabela_municipios_ibge['nome_municipio'].str.strip()
)

assert tabela_municipios_ibge['id_municipio'].notna().all()
assert tabela_municipios_ibge['id_municipio'].is_unique

# O merge usa uma chave temporária normalizada para não alterar
# o tipo original de id_municipio gravado na base.
codigo_ibge = normalizar_codigo_ibge(municipios_ano['id_municipio'])
assert codigo_ibge.notna().all(), 'id_municipio nulo ou não numérico.'
assert codigo_ibge.between(1_000_000, 9_999_999).all(), (
    'id_municipio fora do padrão IBGE de 7 dígitos.'
)

municipios_com_nome = (
    municipios_ano
    .drop(columns='nome_municipio', errors='ignore')  # permite reexecutar a célula
    .assign(_codigo_ibge=codigo_ibge)
    .merge(
        tabela_municipios_ibge.rename(columns={'id_municipio': '_codigo_ibge'}),
        on='_codigo_ibge',
        how='left',
        validate='many_to_one',
    )
    .drop(columns='_codigo_ibge')
)

# Posiciona nome_municipio logo após id_municipio.
coluna_nome = municipios_com_nome.pop('nome_municipio')
municipios_com_nome.insert(
    municipios_com_nome.columns.get_loc('id_municipio') + 1,
    'nome_municipio',
    coluna_nome,
)

# Auditoria do merge: municipios_ano só é substituída se tudo estiver correto.
sem_nome = municipios_com_nome.loc[
    municipios_com_nome['nome_municipio'].isna(), CHAVES_MUNICIPIO_ANO
]
assert len(municipios_com_nome) == len(municipios_ano)
assert sem_nome.empty, (
    f'{len(sem_nome)} registros sem correspondência na API do IBGE:\n{sem_nome}'
)

municipios_ano = municipios_com_nome

print(f'Tabela IBGE: {len(tabela_municipios_ibge):,} municípios')
print(
    f'Base município-ano: {municipios_ano.shape[0]:,} linhas x '
    f'{municipios_ano.shape[1]} colunas (com nome_municipio)'
)
display(municipios_ano[CHAVES_MUNICIPIO_ANO + ['nome_municipio']].head())

del codigo_ibge, coluna_nome, sem_nome, municipios_com_nome
gc.collect()


Tabela IBGE: 5,571 municípios
Base município-ano: 11,073 linhas x 50 colunas (com nome_municipio)


,ano,id_uf,id_municipio,nome_municipio
0,2024,11,1100015,Alta Floresta D'Oeste
1,2024,11,1100023,Ariquemes
2,2024,11,1100031,Cabixi
3,2024,11,1100049,Cacoal
4,2024,11,1100056,Cerejeiras


0

## 11. Persistência da base pronta para o notebook de Pipeline

A base consolidada na granularidade **município-ano** é salva ao final desta preparação em uma pasta dedicada no Google Drive. O próximo notebook deverá começar diretamente por esse arquivo, sem reler ou reagregar a base aluno-ano.

O arquivo preserva os **nulos remanescentes nas features**. Isso é intencional: imputação, encoding e scaling serão ajustados somente dentro dos pipelines de Machine Learning e dos folds de validação de 2024, evitando data leakage.

**Arquivo de passagem entre notebooks:** `dados_modelagem_municipio_ano_2024_2025.parquet`.


In [33]:
# ------------------------------------------------------------
# Salva a base pronta para o próximo notebook de Pipeline
# ------------------------------------------------------------

# Usa a variável AMBIENTE_COLAB para checar onde salvar os dados
if AMBIENTE_COLAB:
    DIR_NOTEBOOKS = Path('/content/drive/MyDrive/notebooks')
    DIR_NOTEBOOKS.mkdir(parents=True, exist_ok=True)
    ARQUIVO_DADOS_MODELAGEM = (
        DIR_NOTEBOOKS / 'dados_modelagem_municipio_ano_2024_2025.parquet'
    )
else:
    DIR_MODELAGEM = Path('data/processed/modelagem')
    DIR_MODELAGEM.mkdir(parents=True, exist_ok=True)
    ARQUIVO_DADOS_MODELAGEM = (
        DIR_MODELAGEM / 'dados_modelagem_municipio_ano_2024_2025.parquet'
    )

# Alias simples para deixar explícito que este é o dataframe de passagem
# para o notebook seguinte. Não cria uma cópia adicional em memória.
dados = municipios_ano

# Parquet preserva tipos e nulos e é mais eficiente que CSV para esta base.
dados.to_parquet(
    ARQUIVO_DADOS_MODELAGEM,
    index=False,
    compression='snappy',
)

print('Base pronta para modelagem salva com sucesso.')
print(f'Arquivo: {ARQUIVO_DADOS_MODELAGEM}')
print(f'Dimensões: {dados.shape[0]:,} linhas × {dados.shape[1]:,} colunas')
print(f'Anos: {sorted(dados["ano"].dropna().unique().tolist())}')
print(f'Memória do dataframe: {dados.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

# O alias `dados` é mantido em memória porque representa a saída deste notebook.
gc.collect()


Base pronta para modelagem salva com sucesso.
Arquivo: data/processed/modelagem/dados_modelagem_municipio_ano_2024_2025.parquet
Dimensões: 11,073 linhas × 50 colunas
Anos: [2024, 2025]
Memória do dataframe: 4.51 MB


0

## 12. NOTAS

O arquivo `dados_modelagem_municipio_ano_2024_2025.parquet` é a **entrada oficial do próximo notebook de Pipeline de Machine Learning**.

O próximo notebook deverá iniciar diretamente com:

```python
from pathlib import Path
import pandas as pd

ARQUIVO_DADOS_MODELAGEM = Path(
    '/content/drive/MyDrive/notebooks/dados_modelagem_municipio_ano_2024_2025.parquet'
)

dados = pd.read_parquet(ARQUIVO_DADOS_MODELAGEM)
```

Nenhuma imputação, codificação ou normalização foi aprendida antecipadamente neste notebook.
